In [34]:
import numpy as np
from typing import List, Dict, Union, Tuple
from tenpy.networks.site import SpinSite
from tenpy.linalg import np_conserved as npc
from tenpy.linalg.np_conserved import Array
from tenpy.networks.mpo import MPO
from tenpy.models.model import MPOModel
from tenpy.models.lattice import Chain
from tenpy.networks.mps import MPS  
from tenpy.algorithms.network_contractor import ncon

In [35]:
def _dirichlet_weight(rng: np.random.Generator, n: int, alpha: float = 1.0) -> np.ndarray:
    """
    Return w ∈ R^n, w ≥ 0, sum(w) = 1.
    Samples w via independent Gamma(alpha,1) and normalizing.
    - alpha = 1.0  → uniform over the simplex (flat with no bias).
    - alpha < 1.0  → spikier (few large entries, many near zero).
    - alpha > 1.0  → smoother (entries more even).
    Edge cases:
      n = 0 → empty vector, n = 1 → [1.0].
    """

    #Sanity checks to avoid useless computation (R^0 and R vectors)
    if n == 0:
        return np.empty((0,), dtype=np.float64)
    if n == 1:
        return np.array([1.0], dtype=np.float64)
    #Invalid parameter
    if alpha <= 0:
        raise ValueError("dirichlet_alpha must be > 0.")
    
    
    #Gamma sampled vector of R^n
    g = rng.gamma(shape=alpha, scale=1.0, size=n)
    
    #Sum of the vector 
    s = g.sum()
    
    # numerically, s > 0 almost surely for alpha>0; still guard:
    if s == 0.0:
        # extremely unlikely; fallback to a one-hot at a random index
        w = np.zeros(n, dtype=np.float64)
        w[rng.integers(n)] = 1.0
        return w
    return g / s


# --------- Validators (keep constraints honest) ----------

def _check_L_row_substochastic(L: np.ndarray, atol: float = 1e-12):
    # For each (x,i): sum_j L[x,i,j] ≤ 1
    row_sums = L.sum(axis=2)  # (d, Dl)
    if np.any(row_sums > 1 + atol):
        raise AssertionError("random_L: found a row with sum > 1.")
    if np.any(L < -atol) or np.any(L > 1 + atol):
        raise AssertionError("random_L: entries must lie in [0,1].")

def _check_R_col_substochastic(R: np.ndarray, atol: float = 1e-12):
    # For each (x,j): sum_i R[x,i,j] ≤ 1
    col_sums = R.sum(axis=1)  # (d, Dr)
    if np.any(col_sums > 1 + atol):
        raise AssertionError("random_R: found a column with sum > 1.")
    if np.any(R < -atol) or np.any(R > 1 + atol):
        raise AssertionError("random_R: entries must lie in [0,1].")

def _check_box_0_1(A: np.ndarray, atol: float = 1e-12):
    if np.any(A < -atol) or np.any(A > 1 + atol):
        raise AssertionError("random_A1: entries must lie in [0,1].")

# --------- Substochastic generators (switched constraints) ---------
def random_L(
    d: int, Dl: int, Dr: int, seed=None, *,
    dirichlet_alpha: float = 1.0, check: bool = True
) -> np.ndarray:
    """
    Substochastic L_k, shape (d, Dl, Dr).
    Constraint : for each physical index x and each ROW i,
        sum_j L[x, i, j] ≤ 1  (row-substochastic)
    Construction: for each (x,i), draw a total mass s ~ U[0,1) and a Dirichlet
    weight w over Dr entries; set the row L[x,i,:] = s * w.
    """
    rng = np.random.default_rng(seed)
    L = np.zeros((d, Dl, Dr), dtype=np.float64)
    
    #Avoid unnecessary computation if degenerate input
    if d == 0 or Dl == 0 or Dr == 0:
        return L

    #Populate the rows substocastically
    for x in range(d):
        for i in range(Dl):
            s = rng.random()  # total mass in [0,1)
            if s == 0.0:
                continue      # zero row
            w = _dirichlet_weight(rng, Dr, alpha=dirichlet_alpha)  # sums to 1
            print(w)
            print("sum",s)
            L[x, i, :] = s * w                                    # row sum = s ≤ 1

    if check:
        _check_L_row_substochastic(L)
    return L

def random_R(
    d: int, Dl: int, Dr: int, seed=None, *,
    dirichlet_alpha: float = 1.0, check: bool = True
) -> np.ndarray:
    """
    Substochastic R_k, shape (d, Dl, Dr).
    Constraint : for each physical x and each COLUMN j,
        sum_i R[x, i, j] ≤ 1  (column-substochastic)
    Construction: for each (x,j), draw mass s ~ U[0,1) and a Dirichlet over Dl
    entries; set the column R[x, :, j] = s * w.
    """
    rng = np.random.default_rng(seed)
    R = np.zeros((d, Dl, Dr), dtype=np.float64)
    
    #Avoid unnecessary computation if degenerate input
    if d == 0 or Dl == 0 or Dr == 0:
        return R
    
    #Populate the columns substocastically
    for x in range(d):
        for j in range(Dr):
            s = rng.random()  # total mass in [0,1)
            if s == 0.0:
                continue      # zero column
            w = _dirichlet_weight(rng, Dl, alpha=dirichlet_alpha)  # sums to 1
            R[x, :, j] = s * w                                    # column sum = s ≤ 1

    if check:
        _check_R_col_substochastic(R)
    return R

def random_A1(d: int, D: int, seed=None, *, check: bool = True) -> np.ndarray:
    """
    First site core A1, shape (d, 1, D).
    Continuous in [0,1] (uniform). To have occasional exact 1.0 values,
    we can optionally snap a tiny fraction to 1.0 after sampling.
    """
    rng = np.random.default_rng(seed)
    A = rng.uniform(0.0, 1.0, size=(d, 1, D)).astype(np.float64)  # in [0,1)
    
    # Optional: snap some values to 1.0 with tiny prob (disabled by default)
    # snap_mask = rng.random(A.shape) < 1e-4
    # A[snap_mask] = 1.0
    
    if check:
        _check_box_0_1(A)
    return A

def wrap_site_tensor(T: np.ndarray):
    """(d, Dl, Dr) -> npc.Array labels ['vL','p','vR'] shape (Dl, d, Dr)"""
    Ai = np.transpose(T, (1, 0, 2))  # (Dl, d, Dr)
    return npc.Array.from_ndarray_trivial(Ai, labels=['vL', 'p', 'vR'])

def tenpy_sites_and_svs(d: int, right_dims):
    N = len(right_dims)
    S = (d - 1) / 2.0
    site = SpinSite(S=S, conserve=None)
    lattice = Chain(L=N, site=site)
    sites = lattice.mps_sites()
    svs = [np.ones(1, dtype=float)]
    svs += [np.ones(right_dims[i], dtype=float) for i in range(N - 1)]
    svs += [np.ones(1, dtype=float)]
    return sites, svs

In [36]:
def site_tensor_from_Bi(Bi: dict, d: int, *, strict_keys: bool = True):
    """
    Costruisce T[x] = Bi[x] con fallback a zeri. T shape: (d, Dl, Dr).

    Se strict_keys=True, controlla che tutte le chiavi di Bi siano in [0, d-1].
    Richiede che Bi non sia vuoto.
    """
    if not Bi:
        raise ValueError("Bi non può essere vuoto: serve almeno una matrice per fissare (Dl, Dr).")

    if strict_keys:
        bad_keys = [k for k in Bi.keys() if not (0 <= k < d)]
        if bad_keys:
            raise ValueError(f"Chiavi fuori range [0, {d-1}]: {bad_keys}")

    # tutte le matrici devono avere la stessa shape 2D
    shapes = {np.shape(M) for M in Bi.values()}
    if len(shapes) != 1:
        raise ValueError(f"shape incoerenti in Bi: {shapes}")
    Dl, Dr = next(iter(shapes))
    if len((Dl, Dr)) != 2:
        raise ValueError(f"Le matrici in Bi devono essere 2D, shape trovata: {Dl, Dr}")

    # dtype comune (considera anche i fallback a 0)
    dtype = np.result_type(*[np.asarray(Bi.get(x, 0)).dtype for x in range(d)])

    T = np.zeros((d, Dl, Dr), dtype=dtype)
    zero_block = np.zeros((Dl, Dr), dtype=dtype)

    for x in range(d):
        Mx = np.asarray(Bi.get(x, zero_block), dtype=dtype)
        if Mx.shape != (Dl, Dr):
            raise ValueError(f"Bi[{x}] ha shape {Mx.shape}, atteso {(Dl, Dr)}")
        T[x] = Mx

    return T, Dl, Dr

def build_mps(B_list, d: int):
    # Controllo d intero positivo
    if d < 1 or int(d) != d:
        raise ValueError("`d` deve essere intero positivo.")
    d = int(d)

    N = len(B_list)
    if N == 0:
        raise ValueError("B_list non può essere vuoto.")

    tensors = []
    right_dims = []

    prev_Dr = None

    for i, Bi in enumerate(B_list):
        T, Dl, Dr = site_tensor_from_Bi(Bi, d)

        # Bordo sinistro
        if i == 0:
            if Dl != 1:
                raise ValueError(f"sito {i}: Dl deve essere 1 (bordo sinistro), trovato {Dl}")
        else:
            # Bond interno: Dl(i) deve coincidere con Dr(i-1)
            if Dl != prev_Dr:
                raise ValueError(
                    f"mismatch bond interno tra sito {i-1} e {i}: "
                    f"Dr[{i-1}]={prev_Dr}, Dl[{i}]={Dl}"
                )

        # Bordo destro
        if i == N - 1 and Dr != 1:
            raise ValueError(f"sito {i}: Dr deve essere 1 (bordo destro), trovato {Dr}")

        tensors.append(T)
        right_dims.append(Dr)
        prev_Dr = Dr

    A = [wrap_site_tensor(T) for T in tensors]
    sites, svs = tenpy_sites_and_svs(d, right_dims)

    return MPS(sites, A, svs, bc='finite', form='A')

In [37]:
def build_B_list(S0, K, N, d_op, m_op, u_op, pd, pu):
    pmid = 1 - pd - pu
    if not (0 <= pd <= 1 and 0 <= pu <= 1 and pd + pu <= 1):
        raise ValueError("Probabilità non valide: servono pd, pu >=0 e pd+pu <= 1")

    B_list = []

    # Sito 1: (1,2)
    B1 = {
        0: np.array([[ (S0/(N))*d_op*pd,  (S0/(N))*d_op*pd - K*pd ]]),
        1: np.array([[ (S0/(N))*m_op*pmid, ((S0/(N))*m_op - K)*pmid ]]),
        2: np.array([[ (S0/(N))*u_op*pu,  (S0/(N))*u_op*pu - K*pu ]]),
    }
    B_list.append(B1)

    # Siti 2..N-1: (2,2)  ← QUI MANCANO NEL TUO FILE
    Bi = {
            0: np.array([[ d_op*pd,  d_op*pd ],
                         [     0,                               pd ]]),
            1: np.array([[ m_op*pmid, m_op*pmid ],
                         [      0,                               pmid ]]),
            2: np.array([[ u_op*pu,  u_op*pu],
                         [    0,                                 pu ]]),
        }
    for i in range(2, N):
        B_list.append(Bi)

    # Sito N: (2,1)
    BN = {
        0: np.array([[ d_op*pd ],
                     [     pd ]],),
        1: np.array([[ m_op*pmid ],
                     [      pmid ]]),
        2: np.array([[ u_op*pu ],
                     [     pu ]]),
    }
    B_list.append(BN)
    return B_list


In [38]:

def build_mps_AR_bond(d: int, N: int, D: int, seed=None):
    """
    Crea un MPS psi con N siti (N dispari) e bond che crescono verso il centro
    come Dl -> min(Dl * d, D) e poi decrescono in modo simmetrico.

    Tensore sito i (0-based) ha shape (d, Dl_i, Dr_i).
    """
    # controlli basilari
    if N < 3 or N % 2 == 0:
        raise ValueError("N deve essere dispari e >= 3.")
    if d < 1 or int(d) != d:
        raise ValueError("`d` deve essere intero positivo.")
    if D < 1 or int(D) != D:
        raise ValueError("`D` deve essere intero positivo.")
    d = int(d)
    D = int(D)

    # half = indice del bond centrale
    half = N // 2  # per N=7 -> 3

    # costruiamo i bond da sinistra fino al centro
    bonds = [1]   # bond[0] = 1
    Dl = 1
    for k in range(1, half + 1):
        candidate = Dl * d
        Dr = candidate if candidate < D else D
        bonds.append(Dr)
        Dl = Dr

    # riflettiamo per avere simmetria: [1, ..., centro, ..., 1]
    # es: [1,3,9,10] -> [1,3,9,10,10,9,3,1]
    bonds = bonds + bonds[::-1]
    assert len(bonds) == N + 1  # N siti -> N+1 bond

    tensors = []

    # primo sito: Dl = bonds[0]=1, Dr = bonds[1]
    Dr0 = bonds[1]
    A1 = random_A1(d, Dr0, seed=seed)   # (d,1,Dr0)
    tensors.append(A1)

    # siti 1..N-1
    for i in range(1, N):
        Dl_i = bonds[i]
        Dr_i = bonds[i + 1]
        sk = None if seed is None else seed + i
        tensors.append(random_R(d, Dl=Dl_i, Dr=Dr_i, seed=sk))  # (d, Dl_i, Dr_i)

    # wrap TenPy
    A = [wrap_site_tensor(T) for T in tensors]  # (d,Dl,Dr) -> (Dl,d,Dr)
    right_dims = [T.shape[2] for T in tensors]  # Dr per sito
    sites, svs = tenpy_sites_and_svs(d, right_dims)
    return MPS(sites, A, svs, bc='finite', form='A')

In [39]:
S0, K, N = 100, 10, 11
d_op, m_op, u_op = 0.9, 1.0, 1.1
pd, pu = 0.25, 0.25   

B_list = build_B_list(S0, K, N, d_op, m_op, u_op, pd, pu)

b = build_mps(B_list, d=3)

for k in range(1,b.L-1):
    Bk = b.get_B(k).to_ndarray()
    print(f"Site {k}, shape = {Bk.shape}:\n{Bk.transpose(1,0,2)}\n")
    print(f"PROVA RESHAPE. \n{Bk.reshape(2,3*2).reshape(2,3,2)}")

#A_prime.reshape(Dl * d_phys, Dr)
psi = build_mps_AR_bond(d=3, N=len(B_list), D=5)
for k in range(psi.L):
    psik = psi.get_B(k).to_ndarray()
    #print(f"Site {k}, shape = {psik.shape}:\n{psik.transpose(1,0,2)}\n")

val = b.overlap(psi)
print("sum_x psi(x)b(x) =", val)

Site 1, shape = (2, 3, 2):
[[[0.225 0.225]
  [0.    0.25 ]]

 [[0.5   0.5  ]
  [0.    0.5  ]]

 [[0.275 0.275]
  [0.    0.25 ]]]

PROVA RESHAPE. 
[[[0.225 0.225]
  [0.5   0.5  ]
  [0.275 0.275]]

 [[0.    0.25 ]
  [0.    0.5  ]
  [0.    0.25 ]]]
Site 2, shape = (2, 3, 2):
[[[0.225 0.225]
  [0.    0.25 ]]

 [[0.5   0.5  ]
  [0.    0.5  ]]

 [[0.275 0.275]
  [0.    0.25 ]]]

PROVA RESHAPE. 
[[[0.225 0.225]
  [0.5   0.5  ]
  [0.275 0.275]]

 [[0.    0.25 ]
  [0.    0.5  ]
  [0.    0.25 ]]]
Site 3, shape = (2, 3, 2):
[[[0.225 0.225]
  [0.    0.25 ]]

 [[0.5   0.5  ]
  [0.    0.5  ]]

 [[0.275 0.275]
  [0.    0.25 ]]]

PROVA RESHAPE. 
[[[0.225 0.225]
  [0.5   0.5  ]
  [0.275 0.275]]

 [[0.    0.25 ]
  [0.    0.5  ]
  [0.    0.25 ]]]
Site 4, shape = (2, 3, 2):
[[[0.225 0.225]
  [0.    0.25 ]]

 [[0.5   0.5  ]
  [0.    0.5  ]]

 [[0.275 0.275]
  [0.    0.25 ]]]

PROVA RESHAPE. 
[[[0.225 0.225]
  [0.5   0.5  ]
  [0.275 0.275]]

 [[0.    0.25 ]
  [0.    0.5  ]
  [0.    0.25 ]]]
Site 5, shape = 

In [40]:
def contract_left(tensor, i, N):
    A = [tensor.get_B(k) for k in range(i-1)]   # each: (Dl, d, Dr)
    #print(len(A))

    if i == 1:
        return None   # oppure semplicemente "return" per terminare senza valore

    if i > N:
        raise ValueError(f"Indice i={i} maggiore del numero di siti N={N}.")
    
    if i <= 0:
        raise ValueError(f"Indice i={i} minore o uguale a 0.")


    # ncon index bookkeeping:
    # open legs use negative integers; positive integers are contracted
    # A0: (-1, -2,  1)
    # A1: ( 1, -3,  2)
    # A2: ( 2, -4,  3)
    # ...
    # A_{i-2}: (i-2, -(i), -(i+1))  # the final right bond stays open

    con_tensors = []
    con_indices = []
    if len(A) == 1:
        con_indices.append([-1, -2, -3])
        con_tensors.append(A[0])
    else:
        for k, Ak in enumerate(A):
            if k == 0:                                  # first site
                con_tensors.append(Ak)
                con_indices.append([-1, -2, 1])
            elif k < len(A) - 1:                        # middle sites
                con_tensors.append(Ak)
                con_indices.append([k, -(k+2), k+1])
            else:                                       # last of the block
                con_tensors.append(Ak)
                con_indices.append([k, -(k+2), -(k+3)])
    #print(con_indices)

    # result has open legs: [-1 (left=1), -2..-(i) (physicals), -(i+1) (right bond Di-1)]
    T = ncon(con_tensors, con_indices)
    # T.shape == (1, d, d, ..., d, D_{i-1})   # (i-1) copies of d

    return T

def contract_right(tensor, i, N):
    A = [tensor.get_B(k) for k in range(i,N)]   # each: (Dl, d, Dr)
    #print(len(A))

    if i == N:
        return None   # oppure semplicemente "return" per terminare senza valore

    if i > N:
        raise ValueError(f"Indice i={i} maggiore del numero di siti N={N}.")
    
    if i <= 0:
        raise ValueError(f"Indice i={i} minore o uguale a 0.")

    # ncon index bookkeeping:
    # open legs use negative integers; positive integers are contracted
    # Ai: (-1, -2,  1)
    # Ai+1: ( 1, -3,  2)
    # Ai+2: ( 2, -4,  3)
    # ...
    # AN-1: (i-2, -(i), -(i+1))  # the final right bond stays open

    con_tensors = []
    con_indices = []
    if len(A) == 1:
        con_indices.append([-1, -2, -3])
        con_tensors.append(A[0])
    else:
        for k, Ak in enumerate(A):
            if k == 0:                                  # first site
                con_tensors.append(Ak)
                con_indices.append([-1, -2, 1])
            elif k < len(A) - 1:                        # middle sites
                con_tensors.append(Ak)
                con_indices.append([k, -(k+2), k+1])
            else:                                       # last of the block
                con_tensors.append(Ak)
                con_indices.append([k, -(k+2), -(k+3)])
    #print(con_indices)

    # result has open legs: [-1 (left=1), -2..-(i) (physicals), -(i+1) (right bond Di-1)]
    T = ncon(con_tensors, con_indices)
    # T.shape == (1, d, d, ..., d, D_{i-1})   # (i-1) copies of d
    return T

UL = contract_left(tensor=psi,i=3,N=len(B_list))
UR = contract_right(tensor=psi,i=3,N=len(B_list))

In [41]:
def tensorial_derivative(psi, b, site):

    N_psi = psi.L
    N_b  = b.L
    if N_psi != N_b:
        raise ValueError(f"psi e b hanno lunghezze diverse: {N_psi} vs {N_b}")
    N = N_psi

    if not (1 <= site <= N):
        raise ValueError(f"site={site} fuori range [1, {N}]")
    
    UL = contract_left(tensor=psi,i=site,N=N)
    UR = contract_right(tensor=psi,i=site,N=N)
    BL = contract_left(tensor=b,i=site,N=N)
    BR = contract_right(tensor=b,i=site,N=N)

    if UL is not None and len(UL.shape) != len(BL.shape):
        raise ValueError(f"Dimensione di UL diversa da BL")
    if UR is not None and len(UR.shape) != len(BR.shape):
        raise ValueError(f"Dimensione di UR diversa da BR")

    if UL is not None:
        left_tensors = [UL,BL]
        left_links = [
            [-1] + [k for k in range(1,site)] + [-2], 
        [-3] + [k for k in range(1,site)] + [-4] 
        ]
        #print('left contraction links',left_links)
        left_contraction = ncon(left_tensors,left_links) 
    ## Il risultato è un tensore di dimensione (1,D_i;1,2)

    if UR is not None:
        right_tensors = [UR,BR]
        right_links = [
            [-1] + [k for k in range(1,N-site+1)] + [-2], 
        [-3] + [k for k in range(1,N-site+1)] + [-4] 
        ]
        #print('right contraction links',right_links)
        right_contraction = ncon(right_tensors,right_links) 
    ## Il risultato è un tensore di dimensione (D_i+1,1;2,1)

    if site > 1 and site < N:
        final_tensors = [left_contraction, b.get_B(site-1),right_contraction]
        final_links = [
            [-1] + [-2] + [-3] + [1],
            [1] + [-4] + [2],
            [-5] + [-6] + [2] + [-7]
        ]
        final_contraction = ncon(final_tensors,final_links)
        #print(final_links)
        #print(final_contraction.shape)

    elif site == 1:
        final_tensors = [b.get_B(site-1),right_contraction]
        final_links = [
            [-1] + [-2] + [1],
            [-3] + [-4] + [1] + [-5]
        ]
        final_contraction = ncon(final_tensors,final_links)
        #print(final_links)
        #print(final_contraction.shape)

    elif site == N:
        final_tensors = [left_contraction, b.get_B(site-1)]
        final_links = [
            [-1] + [-2] + [-3] + [1],
            [1] + [-4] + [-5],
        ]
        final_contraction = ncon(final_tensors,final_links)
        #print(final_links)
        #print(final_contraction.shape)

    return np.squeeze(final_contraction)  

In [42]:
def Greedy(A):
    M=A
    Mrows=M.shape[0]
    L=np.identity(Mrows)
    
    
    while L.shape[1]>A.shape[1]:
        rowvals=np.zeros(M.shape[0])
        mergevals=np.full((M.shape[0], M.shape[0]), -np.inf, dtype=float)
        for i in range(M.shape[0]):
            rowvals[i]=M[i][M[i] > 0].sum()

        for i in range(M.shape[0]):
            for j in range(M.shape[0]):
                if i==j:
                    continue
                s = M[i] + M[j] 
                pos_sum = s[s > 0].sum()
                #print("sum of merged",pos_sum)
                #print("rowvals i ",rowvals[i])
                #print("rowvals j ",rowvals[j])
                mergevals[i, j] = pos_sum - rowvals[i] - rowvals[j]
                #print(mergevals[i,j])
            
        #print(mergevals)
        ropt=np.argmin(rowvals)
        iopt, jopt = np.unravel_index(mergevals.argmax(), mergevals.shape)
        #print("rowvals :",rowvals)
        #print("mergevalsvals :",mergevals)
        #print("iopt",iopt)
        #print("jopt",jopt)
        if mergevals[iopt,jopt]>-rowvals[ropt]:
            #print("sto mergiando")
            M[iopt]=M[iopt]+M[jopt]
            M=np.delete(M,jopt,axis=0)
            L[:,iopt]=L[:,iopt]+ L[:,jopt]
            L=np.delete(L,jopt,axis=1)

        else:
            M=np.delete(M,ropt,axis=0)
            L=np.delete(L,ropt,axis=1)

    return L 

In [43]:
def SweepingAlgorithm(b, d, D, err):
    N = b.L
    psi = build_mps_AR_bond(d=d, N=N, D=D)

    # overlap iniziale
    k_ref = b.overlap(psi)
    print(k_ref)

    # primo update sul sito 1 (come prima)
    site = 1
    A_tilde = tensorial_derivative(psi=psi, b=b, site=site)
    A_prime = A_tilde.to_ndarray()
    L_out = Greedy(A_prime)
    d_phys, Dr = L_out.shape
    L = L_out.reshape(1, d_phys, Dr)
    L_tenpy = npc.Array.from_ndarray_trivial(L, labels=['vL', 'p', 'vR'])
    psi.set_B(site - 1, L_tenpy)

    nstep = 1
    updates_since_check = 1   # abbiamo già fatto un update
    K = 4        # o N, 2*N, ecc.
    max_steps = 100000

    while nstep < max_steps:

        # sweep left -> right
        for i in range(N - 1):
            site = i + 1
            if site == 1 and nstep == 1:
                continue

            A_prime = tensorial_derivative(psi=psi, b=b, site=site).to_ndarray()

            if 1 < site < N:
                Dl, d_phys, Dr = A_prime.shape
                L = Greedy(A_prime.transpose(1,0,2).reshape(Dl * d_phys, Dr)).reshape(d_phys, Dl, Dr).transpose(1,0,2)
            else:
                d_phys, Dr = A_prime.shape
                L =  Greedy(A_prime).reshape(1, d_phys, Dr)

            psi.set_B(i, npc.Array.from_ndarray_trivial(L, labels=['vL', 'p', 'vR']))

            nstep += 1
            updates_since_check += 1

            if updates_since_check >= K:
                k_curr = b.overlap(psi)
                print(k_curr)
                if np.abs(k_curr - k_ref) <= err * max(np.abs(k_ref), 1e-12):
                    return psi, k_curr, nstep
                k_ref = k_curr
                updates_since_check = 0

        # sweep right -> left
        for j in range(N - 1, 0, -1):
            site = j + 1
            A_prime = tensorial_derivative(psi=psi, b=b, site=site).to_ndarray()

            if 1 < site < N:
                Dl, d_phys, Dr = A_prime.shape
                R = Greedy(A_prime.reshape(Dl, Dr * d_phys).T).T.reshape(Dl, d_phys, Dr)
            else:
                Dl, d_phys = A_prime.shape
                R = Greedy(A_prime.T).T.reshape(Dl, d_phys, 1)

            psi.set_B(j, npc.Array.from_ndarray_trivial(R, labels=['vL', 'p', 'vR']))

            nstep += 1
            updates_since_check += 1

            if updates_since_check >= K:
                k_curr = b.overlap(psi)
                print(k_curr)
                if np.abs(k_curr - k_ref) <= err * max(np.abs(k_ref), 1e-12):
                    return psi, k_curr, nstep
                k_ref = k_curr
                updates_since_check = 0

    raise RuntimeError("SweepingAlgorithm did not converge within max_steps")

In [44]:
S0, K, N = 100, 10, 11
d_op, m_op, u_op = 0.9, 1.0, 1.1
pd, pu = 0.25, 0.25   
B_list = build_B_list(S0, K, N, d_op, m_op, u_op, pd, pu)
b = build_mps(B_list, d=3)
psi , K, nstep = SweepingAlgorithm(b=b, d=3, D=10, err=1e-18)
print(nstep)

0.032886052008756765
0.04402047357031282
1.0716865348675713
11.219654906150616
9.237269586086208
32.5188139893979
7.749351746552353
12.315479383513521
9.166718524424173
11.161971527384434
29.323053291807085
7.1123196251445595
9.515965642794963
11.502399668993263
9.93792037044754
23.155205165628917
11.424785158078192
9.247867867283277
9.607916251108508
12.198031317815545
36.212098215757436
10.952363730962436
10.799775333826819
8.142826164197645
24.560670000480123
33.818655829716405
17.271938937484865
10.153549159367957
9.500647182419208
11.381643752361123
26.746752871217044
12.539168002053266
12.327449426495045
9.03872821184722
10.51406875515941
27.13451224419824
12.067317943239102
10.15013548489706
10.106235232714642
10.585249292373145
30.377333638027118
13.20241769895215
8.193497310896033
8.83850015834468
12.698363615138685
26.800442932393818
13.337439609869005
7.444354035188898
9.77455085003187
8.657636562180349
21.907194700671436
7.522611553202237
10.273053367043019
9.96588467065639

KeyboardInterrupt: 